# RoPE Frequency-Band Ablation — Kaggle runner

Settings: **Accelerator = GPU T4 x2** (not P100), **Internet = On**.
Use *Save Version → Save & Run All (Commit)* for the long runs: it runs in the background (≤12 h) and keeps `/kaggle/working` as output.

**Resume across sessions:** add the previous version's output as an input (*Add Input → Your Work → this notebook*). Cell 2 copies its `results/` back, and the runner skips finished cells.

Run phase by phase. Stop if a gate fails.

In [ ]:
# 1. Setup — edit these
REPO_URL = "https://github.com/leeduy0403/rope-freq-ablation.git"
PRIVATE_REPO = False      # True -> add a Kaggle secret named GITHUB_TOKEN
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

import os, subprocess
url = REPO_URL
if PRIVATE_REPO:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
if not os.path.exists("/kaggle/working/rope-freq-ablation"):
    subprocess.run(["git", "clone", "-q", url, "/kaggle/working/rope-freq-ablation"], check=True)
os.chdir("/kaggle/working/rope-freq-ablation")
!git log --oneline -1
!pip install -q pyyaml
M = f"--model-id {MODEL_ID}"


In [ ]:
# 2. Restore results from a previous version (if attached as input)
import glob, shutil
for src in glob.glob("/kaggle/input/*/rope-freq-ablation/results") + glob.glob("/kaggle/input/*/results"):
    print("restoring from", src)
    shutil.copytree(src, "results", dirs_exist_ok=True)


In [ ]:
# Phase 0 — environment (gate G0)
!python -m src.env_report
!python -m pytest -q tests


In [ ]:
# Phase 1 — model candidates (config only, no weights) + rotary module inspection
!python -m src.select_model Qwen/Qwen2.5-0.5B-Instruct Qwen/Qwen2.5-1.5B-Instruct Qwen/Qwen2.5-3B-Instruct
!python -m src.rope.patch --inspect {M}


In [ ]:
# Smoke test — 5 examples, shortest length (minutes). Separate results dir.
!python -m src.eval.runner --stage baseline {M} --tasks s_niah vt fwe --lengths 1024 --n 5 --results-dir results/_smoke
!python -m src.eval.runner --stage sweep {M} --tasks s_niah --lengths 1024 --n 5 --bands 0 7 --results-dir results/_smoke


In [ ]:
# Phase 2 — baseline (gate G2: shortest length s_niah>0.6, vt>0.3, fwe>0.3)
!python -m src.eval.runner --stage baseline {M}


In [ ]:
# Phase 3a — patch correctness (gate G3)
!python -m src.check_patch {M} --tasks s_niah vt --n 20


In [ ]:
# Phase 3b — calibrate compress_factor on one task/length; pick the one with readable dynamic range
for f in (2, 8, 32):
    !python -m src.eval.runner --stage sweep {M} --tasks vt --lengths 2048 --factor {f}


In [ ]:
# Phase 3c — full sweep (gate G4: band=none reproduces baseline). Set FACTOR from 3b.
FACTOR = 8
!python -m src.eval.runner --stage sweep {M} --factor {FACTOR}


In [ ]:
# Phase 4/5 — compression profiles, heatmap, overlay, correlations
TAG = f"compress_f{FACTOR}"
!python -m src.analyze.heatmap {M} --tag {TAG}
!python -m src.analyze.overlay {M} --tag {TAG} --scale 4
from IPython.display import Image, display
for p in sorted(glob.glob("figures/*.png")): print(p); display(Image(p))


In [ ]:
# Package results for download
!cd /kaggle/working && zip -qr results.zip rope-freq-ablation/results rope-freq-ablation/figures && ls -la results.zip
